# 05 - Run (production)

The routine path. Set a date range, run top to bottom, get one workbook covering
every fixture across all five leagues.

This notebook contains no modelling logic and never re-derives anything the
research notebooks already solved -- it loads frozen artifacts and scores. It does
not touch Cleaning, Features, Tuning or Evaluation.

In [1]:
# The package is installed editable (`pip install -e .`), so this works from any
# working directory -- no `os.getcwd()` gymnastics.
import fpp
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
print("fpp", fpp.__version__, "| targets:", fpp.TARGETS)

fpp 0.1.0 | targets: ('goals', 'shots', 'sot', 'corners')


## 1. Date range

In [2]:
DATE_FROM = "11-09-2026"     # dd-mm-yyyy
DATE_TO   = "12-09-2026"
REFRESH   = True            # pull fresh current-season data first

## 2. Refresh

Same ingestion functions as the Data Pull notebook -- not a copy of them.

In [3]:
if REFRESH:
    print(fpp.ingest.refresh("current", date_from=DATE_FROM, date_to=DATE_TO))
else:
    print("skipped - using data already on disk")

[understat_current]


[09/11/26 16:58:43] INFO     No custom team name replacements found. You can configure these in       _config.py:91
                             /Users/patrickknott/soccerdata/config/teamname_replacements.json.                     

                    INFO     No custom league dict found. You can configure additional leagues in    _config.py:189
                             /Users/patrickknott/soccerdata/config/league_dict.json.                               

[match_stats]
  Dropped 1,444 event(s) with all-zero stats (ESPN empty-block sentinel, not a 0-0 result)
  Dropped 1,050 event(s) with every shot on target and no corners (malformed ESPN stat block, not a played match)
  Parsed 30,771 events with full shots/SOT/corners from the cache
  90 second-tier-only club(s) unmapped -- expected, Understat does not cover those divisions; their matches are still kept for the mapped opponent
  Saved 29,076 rows -> /Users/patrickknott/Desktop/Football_Prediction_Project V2/Inputs/Stats/espn_match_stats.csv
[reconcile]
  Understat: 6 season(s) to refetch
  Understat league index was 8.9 days old -- refetching (a season that started since it was written is invisible until this)


[09/11/26 16:58:56] INFO     Saving cached data to                                                   _common.py:250
                             /Users/patrickknott/Desktop/Football_Prediction_Project                               
                             V2/Inputs/soccerdata_cache                                                            

...

  Bund: 7 fixtures -> bundesliga_fixtures.csv


[09/11/26 17:00:03] INFO     Saving cached data to                                                   _common.py:250
                             /Users/patrickknott/Desktop/Football_Prediction_Project                               
                             V2/Inputs/soccerdata_cache                                                            

  Serie: 4 fixtures -> serie_a_fixtures.csv


[09/11/26 17:00:16] INFO     Saving cached data to                                                   _common.py:250
                             /Users/patrickknott/Desktop/Football_Prediction_Project                               
                             V2/Inputs/soccerdata_cache                                                            

  Ligue: 6 fixtures -> ligue_1_fixtures.csv
Ingest (current): 5 step(s) ran
  ok      understat_current
  ok      match_stats
  ok      reconcile
  ok      all_comp_current
  ok      fixtures


## 3. Load

In [4]:
tm  = fpp.clean.build_clean_table() if REFRESH else fpp.clean.load_clean_table()
ctx = fpp.RunContext.load()

print(f"history : {len(tm):,} team-rows to {tm['date'].max().date()}")
print(f"artifacts: {ctx.version}")
for t in fpp.TARGETS:
    print(f"  {t:8} L={ctx.windows[t]['L']:>2} alpha={ctx.windows[t]['alpha']:.2f} "
          f"{len(ctx.features[t])} features, {ctx.n_estimators[t]} trees")

  ESPN stats joined to 20,928/21,735 matches (96.3%)
  12 league-season(s) below 95% ESPN coverage:
league_key    season  matches  with_stats  pct
      Bund 2021/2022      306         273 89.2
      Bund      2627       18          16 88.9
      Liga 2022/2023      380         357 93.9
      Liga      2627       41          35 85.4
     Ligue      2627       27          23 85.2
      Prem 2016/2017      380          18  4.7
      Prem 2021/2022      380         353 92.9
      Prem 2022/2023      380         297 78.2
      Prem      2627       30          28 93.3
     Serie 2022/2023      380         342 90.0
     Serie 2023/2024      380         344 90.5
     Serie      2627       30          24 80.0
  Saved 43,470 rows -> /Users/patrickknott/.cache/football_prediction/clean/team_matches_c0458464dbca.parquet
history : 43,470 team-rows to 2026-09-07
artifacts: v2026-08-25
  goals    L=41 alpha=0.02 5 features, 417 trees
  shots    L=50 alpha=0.10 12 features, 170 trees
  sot      L=60 

## 4. Retrain on the full history

Including the seasons held out during research. That period existed to get an
honest read on the model, not to be preserved forever -- once a model has been
evaluated and accepted there is no reason to keep starving it of recent matches.

In [5]:
models = fpp.predict.train_production(tm, ctx)

  goals    trained on 39,818 rows, 5 features, 417 trees
  shots    trained on 38,212 rows, 12 features, 170 trees
  sot      trained on 38,212 rows, 4 features, 101 trees
  corners  trained on 38,212 rows, 4 features, 102 trees


## 5. Score

Upcoming fixtures are appended to the team-match table as rows with unknown
outcomes and pushed through the *same* buffer pass as history. Record-before-update
on such a row is by construction the correct as-of-now prior -- which is why there
is no separate "latest priors" lookup to drift out of sync.

**Read the coverage check.** A fixture team with no history is not an error and
does not stop the run -- it is scored from the league average alone, which looks
exactly like a real prediction and is not one. A team whose history is years old
is the quieter version of the same problem. Neither was visible before.

In [6]:
fixtures = fpp.predict.load_upcoming_fixtures(
    pd.to_datetime(DATE_FROM, format="%d-%m-%Y"),
    pd.to_datetime(DATE_TO, format="%d-%m-%Y"),
)
print(f"{len(fixtures)} fixtures in range")

coverage = fpp.predict.check_fixture_coverage(fixtures, tm)

preds = fpp.predict.score_fixtures(tm, fixtures, models)
cols = ["date", "league", "home_team", "away_team",
        "goals_home", "goals_away", "shots_home", "shots_away", "corners_home", "corners_away"]
display(preds[cols].head(12))

24 fixtures in range
  all 48 fixture teams have history within 2 season(s)


,date,league,home_team,away_team,goals_home,goals_away,shots_home,shots_away,corners_home,corners_away
0,2026-09-11,Bundesliga,Union Berlin,Schalke 04,1.692716,1.004769,16.377512,9.460524,5.687747,3.710996
1,2026-09-11,Ligue 1,Rennes,Marseille,1.653441,1.716246,13.313569,12.994012,5.395682,4.636351
2,2026-09-11,Serie A,Venezia,Fiorentina,1.600807,1.049794,15.294577,11.180562,6.330719,3.565027
3,2026-09-12,Bundesliga,Augsburg,Bayer Leverkusen,1.655276,1.717099,15.209663,14.025529,5.257812,5.078510
4,2026-09-12,Bundesliga,Borussia Dortmund,Paderborn,2.686098,0.718015,18.230051,8.944703,6.393216,3.066610
5,2026-09-12,Bundesliga,FC Cologne,Werder Bremen,1.819599,1.193959,16.159185,10.727434,6.240189,4.073412
6,2026-09-12,Bundesliga,Freiburg,Borussia M.Gladbach,1.812410,1.181493,15.800513,11.666439,5.457133,4.038147
7,2026-09-12,Bundesliga,Hoffenheim,VfB Stuttgart,1.872572,1.778270,16.009384,12.377849,6.411350,4.768889
8,2026-09-12,Bundesliga,Mainz 05,Eintracht Frankfurt,1.798965,1.449390,16.825161,10.527959,5.901406,3.970948
9,2026-09-12,Ligue 1,Auxerre,Nice,1.373752,1.159865,13.496419,10.857732,5.626558,4.418729


## 6. Write the workbook

One file, every league. Colour coding compares each probability to **that league's
own** realised rate -- the model is pooled, the comparison baseline is not.

The same numbers also go out as JSON for the Edge Book app. It is written *here*
rather than in `06_Split` because this is the only notebook that has them:
`06_Split` reads the filled odds form and nothing else by design, so the
scoreline matrix, the 1X2/BTTS markets and the projected rates are not
recoverable there without retraining or a string join back to this workbook.

In [7]:
path = fpp.report.write_workbook(preds, tm, dispersion=ctx.dispersion)
print("\n->", path)

# The Match Board's whole input. Same sheet codes, same ladders, same league
# baselines as the workbook above -- one derivation, two renderings.
pred_json = fpp.report.write_predictions_json(preds, tm, dispersion=ctx.dispersion)
print("->", pred_json)

Wrote 24 fixture sheets -> /Users/patrickknott/Desktop/Football_Prediction_Project V2/Outputs/predictions_2026-09-11.xlsx

-> /Users/patrickknott/Desktop/Football_Prediction_Project V2/Outputs/predictions_2026-09-11.xlsx
Wrote 24 fixtures -> /Users/patrickknott/Desktop/Football_Prediction_Project V2/Outputs/Data/predictions_2026-09-11.json
-> /Users/patrickknott/Desktop/Football_Prediction_Project V2/Outputs/Data/predictions_2026-09-11.json


## 7. Odds capture form

A second workbook, same sheet names as the first, listing **every** proposition
each fixture is priced at -- the whole ladder, not just the half the model likes
-- with one blank column per book in `staking.BOOKS`.

`MIN_MODEL_P` is zero. The filter existed when the form was typed in by hand; the
`fill-odds` skill now populates it from Oddschecker, so breadth is cheap, and the
lines it used to cut are exactly where a book's margin is widest.

Fill it in, then run `06_Split`. The model's own probability travels with each
row, so that one file is all `06_Split` needs -- nothing has to be matched back
to this workbook afterwards.

In [8]:
props = fpp.staking.propositions(preds, ctx.dispersion)
qual = fpp.staking.qualifying(props)

n_books = len(fpp.staking.BOOK_COLUMNS)
print(f"{len(props)} propositions -> {len(qual)} at P >= {fpp.staking.MIN_MODEL_P:.2f} "
      f"({len(qual) / len(props):.1%}, {len(qual) / len(preds):.1f} per match, "
      f"{len(qual) * n_books:,} cells across {n_books} books)")

form = fpp.report.write_odds_form(qual, source=path.name)
print("\n->", form)

1776 propositions -> 1776 at P >= 0.00 (100.0%, 74.0 per match, 10,656 cells across 6 books)
Wrote 1776 propositions across 24 fixtures -> /Users/patrickknott/Desktop/Football_Prediction_Project V2/Outputs/odds_input_2026-09-11.xlsx

-> /Users/patrickknott/Desktop/Football_Prediction_Project V2/Outputs/odds_input_2026-09-11.xlsx
